# 01_02_Data Collection from Statbel and geo.be

In [1]:
from pathlib import Path
from zipfile import ZipFile

import geopandas as gpd
import pandas as pd

import infrabel_punctuality as ip

DATA_DIR = Path("../data/raw/")


# Table of Contents  

- [1. Dictionaries of Datasets to Ingest from Statbel Open Data and the Geo.be GeoPortal](#1-dictionaries-of-datasets-to-ingest-from-statbel-open-data-and-the-geobe-geoportal)
    - [1.1. Dictionary of Datasets from Statbel](#11-dictionary-of-datasets-from-statbel)
    - [1.2. Dictionary of Dataset from Geo.be Geoportal](#12-dictionary-of-dataset-from-geobe-geoportal)  
    
- [2. Statbel and Geo.be Datasets Ingestion](#2-statbel-and-geobe-datasets-ingestion)
    - [2.1. Download Configuration and Execution](#21-download-configuration-and-execution)
    - [2.2. Extraction of the Geographical Data](#22-extraction-of-the-geographical-data)
    - [2.3. Export Verification](#23-export-verification)  


## 1. Dictionaries of Datasets to Ingest from Statbel Open Data and the Geo.be GeoPortal

### 1.1. Dictionary of Datasets from Statbel

* The administrative divisions and population datasets (`communes` and `population`) from the Statbel Open Data portal are only available as `.xlsx` files.

* The `communes` dataset will be merged with the Infrabel `stations` dataset into `dim_station`. It provides the territorial context for each station.

* To optimize performance in Power BI and maintain a Star Schema, the **geographical hierarchy** (Municipality → Province → Region) present in `communes` will be **denormalized** in the *02_02_profiling_and_cleaning_municipalities* notebook.

* The `population` dataset will be used to **enrich `dim_station`** with demographic data (e.g., population density) in the *03_01_building_dimension_station* notebook. This enables punctuality analysis based on the urban environment (major cities, mid-sized towns, or rural areas).

In [2]:
statbel_datasets = {
    "communes" : "https://statbel.fgov.be/sites/default/files/files/opendata/REFNIS%20code/TU_COM_REFNIS-20250101.xlsx",
    "population" : "https://statbel.fgov.be/sites/default/files/files/documents/bevolking/5.11%20Bevolkingsdichtheid/Pop_density_fr.xlsx"
}

### 1.2. Dictionary of Dataset from Geo.be Geoportal

* The territorial division dataset is a GeoPackage (`.gpkg`) that will serve as the spatial reference layer (background map) for Power BI dashboard visualizations.  

* This GeoPackage has the following properties:  
    * **Format**: compressed `.zip`.  
    * **Projection**: Lambert 2008 (EPSG:3812).  
    * **Layer**: as it will be used to render municipal boundaries, the selected layer is `municipality`.  
    * **Source**: geo.be (Belgian Federal Geographic Platform) under Creative Commons 4.0 license.  

In [3]:
geo_be_datasets = {
    "geolocalisation" : "https://ac.ngi.be/remoteclient-open/ngi-standard-open/Vectordata/TerritorialDivisions/TerritorialDivisions-All/1cd00540-b0ca-11ec-a863-186571a04de2_geopackage+sqlite3_3812.zip"
}


## 2. Statbel and Geo.be Datasets Ingestion

### 2.1. Download Configuration and Execution

In [4]:
ip.run_download(
    statbel_datasets, 
    output_dir=DATA_DIR, 
    file_type="xlsx", 
    registry_name="manifest_statbel.txt"
    )

 
Starting download : communes


communes downloaded to ..\data\raw\communes.xlsx
 
Starting download : population


population downloaded to ..\data\raw\population.xlsx


In [5]:
ip.run_download(
    geo_be_datasets, 
    output_dir=DATA_DIR, 
    file_type="zip", 
    registry_name="manifest_geo.txt"
    )

 
Starting download : geolocalisation


geolocalisation downloaded to ..\data\raw\geolocalisation.zip


### 2.2. Extraction of the Geographical Data

The cell below sets the path to the `.zip` file.

In [6]:
zip_path = Path("geolocalisation.zip")

with ZipFile(DATA_DIR / zip_path, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR / zip_path.stem)

The code below displays the list of the decompressed files.  
In this case, the list contains a single file.

In [7]:
final_path = DATA_DIR / zip_path.stem

for file in final_path.iterdir():
    print(file.name)

territorialdivisions_3812.gpkg


### 2.3. Export Verification

In [8]:
municipalities = pd.read_excel(DATA_DIR / "communes.xlsx")
municipalities.head()

,LVL_REFNIS,CD_REFNIS,CD_SUP_REFNIS,TX_REFNIS_DE,TX_REFNIS_FR,TX_REFNIS_NL,DT_VLDT_START,DT_VLDT_END
0,1,2000,-,Flämische Region,Région flamande,Vlaams Gewest,01/01/1970,31/12/9999
1,1,3000,-,Wallonische Region,Région wallonne,Waals Gewest,01/01/1970,31/12/9999
2,1,4000,-,Region Brüssel-Hauptstadt,Région de Bruxelles-Capitale,Brussels Hoofdstedelijk Gewest,01/01/1970,31/12/9999
3,2,10000,02000,Provinz Antwerpen,Province d’Anvers,Provincie Antwerpen,01/01/1970,31/12/9999
4,2,20000,-,Provinz Brabant,Province de Brabant,Provincie Brabant,01/01/1970,31/12/1994


The first row of the Excel sheet is the table title, not the column headers. The actual column headers are on the second row - therefore `header=1`.

In [9]:
population = pd.read_excel(DATA_DIR / "population.xlsx", header=1)
population.head()

,Refnis,Lieu de résidence,Population,Superficie au km²,Habitants / km²
0,1000,Belgique,11431406,30689.368862,372.487491
1,2000,Région flamande,6589069,13625.548930,483.581912
2,3000,Région wallonne,3633795,16901.400088,214.999644
3,4000,Région de Bruxelles-Capitale,1208542,162.419844,7440.851870
4,10000,Province d’Anvers,1857986,2876.120363,646.004258


In [10]:
territorial_divisions = (
    gpd.read_file(final_path / "territorialdivisions_3812.gpkg", layer="municipality")
)
territorial_divisions.head()

,tgid,modifdate,arrondissementcapital,provincecapital,regioncapital,countrycapital,niscode,city,languagestatute,nameger,namefre,namedut,geometry
0,{8BF44CB0-B8FD-44F6-A64F-1307610DA4C9},2007-01-05,False,False,False,False,72004,1,1,Bree,Bree,Bree,"MULTIPOLYGON Z (((735277.943 700725.863 0, 735..."
1,{54A85359-4967-4318-AA63-D234DDED2FD7},2007-01-05,False,False,False,False,63004,0,2,Baelen,Baelen,Baelen,"MULTIPOLYGON Z (((761804.571 647035.961 0, 761..."
2,{95E2AAE2-F9DB-456C-A113-2A21ED6F932F},2007-01-05,False,False,False,False,13003,0,1,Balen,Balen,Balen,"MULTIPOLYGON Z (((708505.572 703629.811 0, 708..."
3,{4487B4B8-4422-4856-97C5-33174BF84028},2007-01-05,False,False,False,False,62011,0,2,Bassenge,Bassenge,Bitsingen,"MULTIPOLYGON Z (((736728.393 660624.972 0, 736..."
4,{68A224E9-B2E7-4225-9FDA-03AE2B9C8C41},2007-01-05,False,False,False,False,85046,0,2,Habay,Habay,Habay,"MULTIPOLYGON Z (((735041.579 544346.475 0, 734..."
